# BRFSS 2024 — Full Pipeline in Google Colab

This notebook runs the **entire** BRFSS mental-health-risk pipeline in Colab by cloning the
GitHub repo (code **and** the committed data come with it), then running each step in order:

1. **Clean** (`BRFSS.py`) &rarr; `BRFSS_Cleaned/BRFSS_2024_cleaned.parquet`
2. **Recode** (`BRFSS_Recode.py`) &rarr; `BRFSS_Cleaned/BRFSS_2024_recoded.parquet`
3. **EDA** (`BRFSS_EDA.py`) &rarr; plots in `BRFSS_EDA/`
4. **Model** (`BRFSS_Model.py`) &rarr; trained models in `BRFSS_Models/`
5. **Evaluate** (`BRFSS_Evaluate.py`) &rarr; metrics + plots in `BRFSS_Evaluation/`

Just **Runtime &rarr; Run all**. Nothing to upload — the data ships with the repo.

> Target: `mental_risk = 1` when `MENTHLTH >= 14` poor-mental-health days in the past 30.

## Step 0 — clone the repo and install dependencies

Safe to re-run: if the repo is already present it just pulls the latest.

In [ ]:
import os

REPO_URL = 'https://github.com/Kudos-4/BRFSS.git'
REPO_DIR = '/content/BRFSS'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}
!ls -la

In [ ]:
# Colab already has pandas / numpy / scikit-learn / matplotlib / seaborn.
# Install the two that may be missing or outdated. Quiet to keep output short.
!pip install -q xgboost pyarrow

## Step 1 — Clean the raw survey

Recodes `88` &rarr; `0` for the health-day counts, drops blanks and "no valid response" codes,
and builds the `mental_risk` target.

In [ ]:
!python BRFSS.py

## Step 2 — Recode into binary features (PI's 5 prep steps)

In [ ]:
!python BRFSS_Recode.py

## Step 3 — Exploratory data analysis

In [ ]:
!python BRFSS_EDA.py

## Step 4 — Train the models

Logistic Regression, Decision Tree, Random Forest, and XGBoost — all class-balanced,
with `MENTHLTH` excluded from the features to avoid target leakage.

In [ ]:
!python BRFSS_Model.py

## Step 5 — Evaluate

In [ ]:
!python BRFSS_Evaluate.py

## Review the results inline

Show the headline metrics table and the key evaluation plots without leaving the notebook.

In [ ]:
import pandas as pd

print('=== Model comparison (training) ===')
display(pd.read_csv('BRFSS_Models/model_comparison.csv', index_col=0))

print('\n=== Test-set metrics ===')
display(pd.read_csv('BRFSS_Evaluation/metrics_table.csv', index_col=0))

In [ ]:
from IPython.display import Image, display
import os

for folder in ['BRFSS_Evaluation', 'BRFSS_EDA']:
    for fname in sorted(os.listdir(folder)):
        if fname.endswith('.png'):
            print(f'{folder}/{fname}')
            display(Image(filename=os.path.join(folder, fname)))

## (Optional) Keep the outputs

Colab discards files when the runtime ends. To save the regenerated recoded dataset, either download it
or copy it to your Google Drive.

In [ ]:
# --- Download the recoded dataset to your computer ---
# from google.colab import files
# files.download('BRFSS_Cleaned/BRFSS_2024_recoded.csv')

# --- Or copy outputs to Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/BRFSS_outputs
# !cp -r BRFSS_Cleaned BRFSS_Models BRFSS_Evaluation BRFSS_EDA /content/drive/MyDrive/BRFSS_outputs/
# print('Copied outputs to Drive.')